# M4 v2 — Resume **final evaluations only** on 2 × A100 SXM GPUs (Runpod)

**Never retrains O/R/U/C. Never regenerates splits. Never rewrites the Colab protocol freeze.** Imports the original Colab project, verifies all 12 trained adapters and all 3 identification gates, and runs **36 complete evaluations = 3 seeds × 4 models × 3 precision conditions** on two independent A100 workers. Uses the original frozen `m4_runner.py` **unchanged**.

### Before starting

In Colab, use `M4_COLAB_EXPORT_TO_RUNPOD.ipynb` to export and download `M4_TRANSFER_CHECKSUMS.json` and **all** `.partNNN` files. In Runpod JupyterLab, run Cell 1 below to create the transfer inbox. In the JupyterLab file browser, enter `/workspace/M4_INCOMING` and upload all those files. Do **not** run the previous `M4_RUNPOD_FLEX...` notebook: its full pipeline starts training again and uses a different freeze layout.

You need one Runpod Pod with **two NVIDIA A100 SXM 80GB cards**, JupyterLab, and approximately 50 GB of volume disk. Once the run finishes, download/export your results before **terminating** the Pod. All compute is billed while the Pod is active.

**Important:** Do not run Colab's Section 9 at the same time. The transfer is taken from your completed Section 8 and Section 7 freeze.

In [ ]:
from pathlib import Path
import os, json, sys, subprocess
INBOX=Path('/workspace/M4_INCOMING');INBOX.mkdir(parents=True,exist_ok=True)
ROOT=Path('/workspace/M4_IMPORTED_V2')
print('Upload your transfer files into',INBOX)
print('Currently present:',[x.name for x in sorted(INBOX.glob('*'))])
print('Runner destination:',ROOT)


## 1. Import and cryptographically verify the exact Colab project

This step checks each transfer part, the full archive, all individual file checksums, the frozen pipeline, every LoRA adapter and each original gate. It will **refuse to overwrite** an existing imported project to protect resumability.

In [ ]:
import hashlib, tarfile, shutil, tempfile

def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()

SEEDS=(20261101,20261102,20261103)
ROLES=('O','R','U','C')
checks_path=INBOX/'M4_TRANSFER_CHECKSUMS.json'
assert checks_path.exists(),f'Missing {checks_path}; upload it and all part files first'
checks=json.loads(checks_path.read_text(encoding='utf-8'))
for name in checks['part_order']:
    path=INBOX/name
    assert path.exists(),f'MISSING transfer part {name}'
    assert path.stat().st_size==checks['parts'][name]['size'],f'Wrong size: {name}'
    assert sha(path)==checks['parts'][name]['sha256'],f'CORRUPTED transfer part {name}'
print('PASS: all',len(checks['part_order']),'transfer parts verified')

if not ROOT.exists():
    combined=INBOX/checks['archive_name']
    if not combined.exists():
        with combined.open('wb') as out:
            for name in checks['part_order']:
                with (INBOX/name).open('rb') as inp:shutil.copyfileobj(inp,out,4*1024*1024)
    assert combined.stat().st_size==checks['archive_size_bytes']
    assert sha(combined)==checks['archive_sha256'],'Full archive SHA mismatch'
    stage=Path(str(ROOT)+'.importing')
    assert not stage.exists(),f'Stale staging dir: {stage}; inspect before reattempting'
    stage.mkdir(parents=True)
    with tarfile.open(combined,'r:gz') as tf:
        for member in tf.getmembers():
            rel=Path(member.name)
            assert rel.parts[0]=='M4_project' and '..' not in rel.parts, f'Unsafe path: {member.name}'
            assert not member.issym() and not member.islnk() and (member.isfile() or member.isdir()), f'Unsafe member: {member.name}'
            dest=stage.joinpath(*rel.parts[1:])
            if member.isdir():dest.mkdir(parents=True,exist_ok=True);continue
            dest.parent.mkdir(parents=True,exist_ok=True)
            with tf.extractfile(member) as source,dest.open('wb') as target:shutil.copyfileobj(source,target)
    stage.rename(ROOT)
    print('Imported project from verified archive')
else:
    print('Existing imported project detected; no files overwritten')

manifest=json.loads((ROOT/'M4_TRANSFER_MANIFEST.json').read_text(encoding='utf-8'))
assert manifest['source_graph_sha256']=='1cf3099355bf17cf5637445907ea4b37d36ffb6301726c671dfbc0cbe2ae095f'
assert sha(ROOT/'mpkg_v1.json')==manifest['source_graph_sha256'],'Published source bytes mismatch'
for rel,meta in manifest['files'].items():
    f=ROOT/rel
    assert f.is_file() and f.stat().st_size==meta['size'] and sha(f)==meta['sha256'],f'Original import file altered or missing: {rel}'
freeze=json.loads((ROOT/'FINAL_TEST_FREEZE.json').read_text(encoding='utf-8'))
assert freeze['version']=='M4_v2_accelerated_20261101'
assert freeze['runner_sha256']==sha(ROOT/'m4_runner.py')==manifest['frozen_runner_sha256']
assert freeze['dataset_manifest_sha256']==sha(ROOT/'M4_CP2_Data'/'manifest.json')==manifest['frozen_manifest_sha256']
assert freeze['model_revision']==(ROOT/'MODEL_REVISION_SHA.txt').read_text().strip()
for seed in SEEDS:
    gate=json.loads((ROOT/'evaluations'/str(seed)/'development_gate.json').read_text())
    assert gate['identified'] is True, f'Original knowledge gate failed: {seed}'
    for role in ROLES:
        p=ROOT/'checkpoints'/str(seed)/role
        m=json.loads((p/'TRAINED.json').read_text())
        assert m['seed']==seed and m['role']==role and m['pipeline_version']==freeze['version']
        assert m['data_manifest_sha256']==freeze['dataset_manifest_sha256']
        assert (p/'adapter'/'adapter_config.json').exists()
        assert list((p/'adapter').glob('*.safetensors'))
print('PASS: original Colab freeze/code/dataset, all 12 adapters, all 3 gates')
print('Gate O recovery:',[json.loads((ROOT/'evaluations'/str(s)/'development_gate.json').read_text())['F_O_dev'] for s in SEEDS])


## 2. Check Runpod GPUs and install the CUDA-compatible quantization environment

Install the original Hugging Face stack and the fixed bitsandbytes version, **without replacing working CUDA-enabled PyTorch**. A100 supports native BF16 while the original Colab T4 used BF16 tensors without native BF16 acceleration; all **final** results here will run on A100 hardware.

In [ ]:
import torch
print('Torch',torch.__version__,'CUDA',torch.version.cuda)
assert torch.cuda.is_available(),'No CUDA GPUs visible; check Runpod Pod'
assert torch.cuda.device_count()==2,f'Expected TWO GPUs on same Pod, found {torch.cuda.device_count()}'
for i in range(2):
    p=torch.cuda.get_device_properties(i)
    print(i,p.name,round(p.total_memory/1024**3,1),'GiB, compute capability',torch.cuda.get_device_capability(i))
    assert 'A100' in p.name and p.total_memory>=70*1024**3,'Expected A100 80GB, not another GPU model'
assert torch.cuda.is_bf16_supported(including_emulation=False),'A100 native BF16 missing'
print('PASS: TWO NATIVE BF16 A100 GPU DEVICES')


In [ ]:
import subprocess,sys
subprocess.run([sys.executable,'-m','pip','install','-q',
    'transformers==4.51.3','peft==0.15.2','accelerate==1.6.0',
    'safetensors>=0.5.0','sentencepiece>=0.2.0','matplotlib>=3.8'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-cache-dir',
    '--force-reinstall','--no-deps','bitsandbytes==0.50.2'],check=True)
print('Dependencies installed; using existing torch and NVIDIA CUDA')


In [ ]:
import subprocess,sys
probe=r"""
import torch,bitsandbytes as bnb
assert torch.cuda.is_available()
x=torch.randn(2,64,device='cuda',dtype=torch.bfloat16)
a=bnb.nn.Linear4bit(64,64,bias=False,compute_dtype=torch.bfloat16,quant_type='nf4').bfloat16().cuda()
b=bnb.nn.Linear8bitLt(64,64,bias=False,has_fp16_weights=False).bfloat16().cuda()
assert torch.isfinite(a(x)).all() and torch.isfinite(b(x)).all()
print('PASS: BF16 NF4 AND INT8 CUDA KERNELS, bitsandbytes',bnb.__version__)
"""
r=subprocess.run([sys.executable,'-c',probe],capture_output=True,text=True)
print(r.stdout[-3000:]);print(r.stderr[-2000:]);assert r.returncode==0,'Packed quantization backend failed; do not start evaluations'


## 3. Create the **evaluation-only** 2-GPU scheduler and preflight all original checkpoint data

The scheduler keeps the **same evaluation settings** as Colab: `--gen-batch 8`, `--nll-batch 8`, `--max-new-tokens 96`, all 360 facts, all 36 model/precision combinations. Jobs are grouped by `(seed, role)` so three precisions reuse one locally merged weight checkpoint on the same GPU. Individual completed evaluations are skipped only if their metadata and row counts match; partial evaluations resume as the original runner defines. No training commands are present.

In [ ]:
RUNNER_SCRIPT = '\\\n#!/usr/bin/env python3\n"""M4 Colab-v2 frozen-protocol continuation, final test ONLY, 2 A100 workers.\nNever trains or changes the original m4_runner.py / FINAL_TEST_FREEZE.json.\n"""\nimport argparse, collections, datetime, hashlib, json, os, queue, subprocess, sys, threading, time, traceback\nfrom pathlib import Path\n\nSEEDS=(20261101,20261102,20261103)\nROLES=(\'O\',\'R\',\'U\',\'C\')\nPRECISIONS=(\'bf16\',\'int8\',\'nf4\')\nLOCK=threading.Lock()\n\ndef log(*v):\n    with LOCK:print(datetime.datetime.now(datetime.timezone.utc).isoformat(timespec=\'seconds\'),*v,flush=True)\n\ndef sha(p):\n    h=hashlib.sha256()\n    with Path(p).open(\'rb\') as f:\n        for b in iter(lambda:f.read(4*1024*1024),b\'\'):h.update(b)\n    return h.hexdigest()\n\ndef read_jsonl(p):\n    return [json.loads(x) for x in Path(p).read_text(encoding=\'utf-8\').splitlines() if x.strip()]\n\ndef preflight(root):\n    import torch\n    if torch.cuda.device_count()!=2:raise RuntimeError(\'Must have exactly two visible GPUs\')\n    if not torch.cuda.is_bf16_supported(including_emulation=False):raise RuntimeError(\'Native BF16 required\')\n    for i in (0,1):\n        p=torch.cuda.get_device_properties(i)\n        if not (\'A100\' in p.name and p.total_memory>=70*1024**3):raise RuntimeError(\'Expected A100 SXM 80GB on both worker devices\')\n    frozen=json.loads((root/\'FINAL_TEST_FREEZE.json\').read_text(encoding=\'utf-8\'))\n    if frozen.get(\'version\')!=\'M4_v2_accelerated_20261101\':raise RuntimeError(\'Unexpected frozen protocol\')\n    runner=root/\'m4_runner.py\'\n    if sha(runner)!=frozen[\'runner_sha256\']:raise RuntimeError(\'Frozen original runner changed\')\n    if sha(root/\'M4_CP2_Data\'/\'manifest.json\')!=frozen[\'dataset_manifest_sha256\']:raise RuntimeError(\'Frozen data manifest changed\')\n    if (root/\'MODEL_REVISION_SHA.txt\').read_text().strip()!=frozen[\'model_revision\']:raise RuntimeError(\'Pinned model revision changed\')\n    if not (root/\'M4_TRANSFER_MANIFEST.json\').is_file():raise RuntimeError(\'Transferred Colab state missing\')\n    for seed in SEEDS:\n        path=root/\'evaluations\'/str(seed)/\'development_gate.json\'\n        g=json.loads(path.read_text(encoding=\'utf-8\'))\n        if g[\'identified\'] is not True:raise RuntimeError(f\'Gate failed for seed {seed}\')\n        for role in ROLES:\n            p=root/\'checkpoints\'/str(seed)/role\n            m=json.loads((p/\'TRAINED.json\').read_text(encoding=\'utf-8\'))\n            if not (m[\'role\']==role and m[\'seed\']==seed and m[\'pipeline_version\']==frozen[\'version\']):raise RuntimeError(\'Invalid trained marker\')\n            if m[\'data_manifest_sha256\']!=frozen[\'dataset_manifest_sha256\']:raise RuntimeError(\'Wrong data marker\')\n            if not any((p/\'adapter\').glob(\'*.safetensors\')):raise RuntimeError(\'Missing trained adapter\')\n        for role in (\'O\',\'R\'):\n            mp=root/\'evaluations\'/str(seed)/\'dev\'/f\'{role}_bf16_meta.json\'\n            if json.loads(mp.read_text(encoding=\'utf-8\'))[\'rows\']!=280:raise RuntimeError(\'Development output missing\')\n    # Evidence that the final test was frozen before all finalized test outputs.\n    expected={\'seed\':None,\'split\':\'test\',\'limit_per_partition\':0,\'gen_batch\':8,\n              \'max_new_tokens\':96,\'full_evaluation\':True}\n    for seed in SEEDS:\n        for role in ROLES:\n            for prec in PRECISIONS:\n                p=root/\'evaluations\'/str(seed)/\'test\'/f\'{role}_{prec}.jsonl\'\n                mp=p.with_name(p.stem+\'_meta.json\')\n                if p.exists() != mp.exists():raise RuntimeError(f\'Completed output/meta mismatch: {p}\')\n                if mp.exists():\n                    m=json.loads(mp.read_text(encoding=\'utf-8\'))\n                    if not all(m.get(k)==v for k,v in expected.items() if k!=\'seed\') or m[\'seed\']!=seed or m[\'role\']!=role or m[\'precision\']!=prec or m[\'rows\']!=360:\n                        raise RuntimeError(f\'Incompatible completed final evaluation: {mp}\')\n                    if \'A100\' not in m.get(\'device\',\'\'):\n                        raise RuntimeError(f\'Completed final evaluation on different hardware: {mp}. Do not mix T4 and A100 final representations.\')\n                    rr=read_jsonl(p)\n                    if len(rr)!=360 or collections.Counter(x[\'partition\'] for x in rr)!=collections.Counter({\'obsolete\':40,\'retain\':240,\'unseen\':80}):\n                        raise RuntimeError(f\'Corrupt completed final result: {p}\')\n    return frozen\n\ndef group_job(root, gpu, seed, role, rev, stop):\n    label=f\'{seed}_{role}\'\n    logs=root/\'runpod_2gpu_logs\';logs.mkdir(parents=True,exist_ok=True)\n    path=logs/f\'{label}_gpu{gpu}.log\'\n    env=os.environ.copy()\n    env.update({\'CUDA_VISIBLE_DEVICES\':str(gpu),\'M4_MODEL_REVISION\':rev,\n          \'M4_LOCAL_MODEL_CACHE\':str(root/\'runpod_cache\'/f\'gpu{gpu}\'),\n          \'HF_HOME\':str(root/\'hf_cache\'),\n          \'TOKENIZERS_PARALLELISM\':\'false\',\'OMP_NUM_THREADS\':\'4\',\'MKL_NUM_THREADS\':\'4\'})\n    for prec in PRECISIONS:\n        if stop.is_set():return\n        completed=root/\'evaluations\'/str(seed)/\'test\'/f\'{role}_{prec}.jsonl\'\n        done_meta=completed.with_name(completed.stem+\'_meta.json\')\n        if done_meta.exists() and completed.exists():\n            # preflight already verified count, metadata, and partition distribution.\n            log(\'SKIP VERIFIED COMPLETE\',f\'{seed}/{role}/{prec}\')\n            continue\n        args=[sys.executable,str(root/\'m4_runner.py\'),\'eval\',\n          \'--data\',str(root/\'M4_CP2_Data\'),\'--out\',str(root),\n          \'--seed\',str(seed),\'--role\',role,\'--split\',\'test\',\'--precision\',prec,\n          \'--limit\',\'0\',\'--gen-batch\',\'8\',\'--nll-batch\',\'8\',\'--max-new-tokens\',\'96\']\n        log(\'START\',f\'{seed}/{role}/{prec}\',\'GPU\',gpu)\n        tic=time.monotonic()\n        with path.open(\'a\',encoding=\'utf-8\') as fp:\n            fp.write(\'\\n=== \'+datetime.datetime.now(datetime.timezone.utc).isoformat()+\' \'+prec+\' ===\\n\')\n            fp.write(\' \'.join(args)+\'\\n\');fp.flush()\n            r=subprocess.run(args,stdout=fp,stderr=subprocess.STDOUT,env=env,cwd=str(root))\n        if r.returncode!=0:\n            stop.set()\n            raise RuntimeError(f\'Failed {seed}/{role}/{prec}, code={r.returncode}, read {path}\')\n        log(\'DONE\',f\'{seed}/{role}/{prec}\',\'GPU\',gpu,\'minutes\',round((time.monotonic()-tic)/60,2))\n\ndef main():\n    a=argparse.ArgumentParser()\n    a.add_argument(\'--root\',type=Path,default=Path(\'/workspace/M4_IMPORTED_V2\'))\n    a.add_argument(\'--preflight\',action=\'store_true\')\n    args=a.parse_args()\n    root=args.root.resolve()\n    freeze=preflight(root)\n    log(\'Verified original Colab v2 freeze. 12 trained adapters and 3 gates. Model revision\',freeze[\'model_revision\'])\n    if args.preflight:return\n    todo=queue.Queue()\n    for seed in SEEDS:\n        for role in ROLES:todo.put((seed,role))\n    stop=threading.Event()\n    errors=[]\n    def worker(gpu):\n        while not stop.is_set():\n            try:seed,role=todo.get_nowait()\n            except queue.Empty:break\n            try:group_job(root,gpu,seed,role,freeze[\'model_revision\'],stop)\n            except Exception as e:\n                log(\'ERROR\',str(e));errors.append(str(e));stop.set()\n            finally:todo.task_done()\n    threads=[threading.Thread(target=worker,args=(g,),daemon=False) for g in (0,1)]\n    for t in threads:t.start()\n    for t in threads:t.join()\n    if errors:raise RuntimeError(\'At least one evaluation failed: \'+str(errors))\n    # Use the EXACT original report command (not rewritten/approximated report calculations).\n    log(\'All 36 combinations processed; running original three-seed report...\')\n    proc=subprocess.run([sys.executable,str(root/\'m4_runner.py\'),\'report\',\n        \'--data\',str(root/\'M4_CP2_Data\'),\'--out\',str(root),\'--min-seeds\',\'3\'],cwd=str(root))\n    if proc.returncode:raise RuntimeError(\'Original M4 report failed\')\n    status=json.loads((root/\'reports\'/\'M4_status.json\').read_text(encoding=\'utf-8\'))\n    if status[\'full_group_count\']!=108 or status[\'full_test_record_count\']!=12960:\n        raise RuntimeError(\'Final completeness check failed: \'+str(status))\n    log(\'PASS: ALL 36 FULL EVALUATIONS, 12,960 OBSERVED RECORDS; REPORTS:\',root/\'reports\')\n    (root/\'RUNPOD_M4_COMPLETE.txt\').write_text(\'COMPLETE 36 FULL EVALUATIONS / 12960 RECORDS\\n\',encoding=\'utf-8\')\n\nif __name__==\'__main__\':main()\n'
SCHEDULER=ROOT/'m4_runpod_resume_2gpu.py'
if SCHEDULER.exists() and SCHEDULER.read_text()!=RUNNER_SCRIPT:
    raise RuntimeError('Existing Runpod launcher differs; refusing to overwrite')
SCHEDULER.write_text(RUNNER_SCRIPT,encoding='utf-8')
import subprocess,sys
subprocess.run([sys.executable,str(SCHEDULER),'--root',str(ROOT),'--preflight'],check=True)
print('PASS: READY TO EVALUATE ONLY — ZERO TRAINING SCHEDULED')

## 4. Start 36 final evaluations on two A100s (detached and resumable)

Starts one **detached** worker process that internally schedules 12 `(seed, role)` groups across both GPUs, evaluates all three precisions for each group, and verifies 12,960 complete records. **Do not run this cell twice while a worker is still active.** Closing Jupyter does not stop the Runpod Pod, but terminating the Pod destroys its Volume Disk.

The original report can contain negative outcomes; it will not be edited to make unlearning look successful.

In [ ]:
import subprocess,sys,json,os,time
LOG=ROOT/'RUNPOD_M4_MASTER.log';PID=ROOT/'RUNPOD_M4_PID.json'

def running(pid):
    try:
        b=(Path('/proc')/str(pid)/'cmdline').read_bytes()
        return b'm4_runpod_resume_2gpu.py' in b
    except (FileNotFoundError,ProcessLookupError):return False

if PID.exists():
    p=json.loads(PID.read_text())['pid']
    assert not running(p),f'ALREADY RUNNING: PID={p}, do not launch again'

with LOG.open('a',encoding='utf-8') as log_file:
    proc=subprocess.Popen([sys.executable,str(SCHEDULER),'--root',str(ROOT)],
      cwd=str(ROOT),stdout=log_file,stderr=subprocess.STDOUT,start_new_session=True,
      env=os.environ.copy())
PID.write_text(json.dumps({'pid':proc.pid,'started_unix':time.time()}))
print('STARTED evaluation-only scheduler PID',proc.pid)
print('Master log:',LOG)
print('Training jobs scheduled: 0')
print('Final eval tasks scheduled: 36 (12 model groups x 3 precisions)')


## 5. Check progress and detailed errors

Rerun this monitoring cell whenever you want. The master log prints task start/done and elapsed minutes. Individual underlying tracebacks are in `runpod_2gpu_logs/`. It is safe to re-run the launcher after a failed run **only if the process is no longer active** and the underlying error has been fixed; the original evaluator resumes partial records.

In [ ]:
import json,subprocess
if LOG.exists():
    print(subprocess.run(['tail','-n','50',str(LOG)],capture_output=True,text=True).stdout)
if (ROOT/'reports'/'M4_status.json').exists():
    print('REPORT STATUS:',(ROOT/'reports'/'M4_status.json').read_text()[:3000])
print('EVALUATIONS / 36:',sum(1 for s in SEEDS for r in ROLES for p in ('bf16','int8','nf4') if
      (ROOT/'evaluations'/str(s)/'test'/f'{r}_{p}_meta.json').exists()))
if PID.exists():
    pid=json.loads(PID.read_text())['pid']
    print('Scheduler running:',running(pid),'PID',pid)


## 6. Export all results before terminating the Pod

Run only **after** `RUNPOD_M4_COMPLETE.txt` exists. This writes a results tarball to `/workspace`. Download the tarball using JupyterLab's file browser and copy it somewhere you control. The Volume Disk disappears when you **terminate** the Pod; merely stopping may still incur storage charges.

In [ ]:
import tarfile,hashlib
assert (ROOT/'RUNPOD_M4_COMPLETE.txt').exists(),'Not complete. Do not export as final yet.'
BACKUP=Path('/workspace/M4_V2_COMPLETE_RESULTS.tar.gz')
with tarfile.open(BACKUP,'w:gz',compresslevel=1) as tf:
    for name in ('reports','evaluations','checkpoints','M4_CP2_Data','FINAL_TEST_FREEZE.json',
                 'MODEL_REVISION_SHA.txt','m4_runner.py','m4_build_dataset.py','mpkg_v1.json','M4_TRANSFER_MANIFEST.json',
                 'RUNPOD_M4_MASTER.log','runpod_2gpu_logs','RUNPOD_M4_COMPLETE.txt'):
        p=ROOT/name
        if p.exists():tf.add(p,arcname='M4_v2/'+name,recursive=True)
print('FINAL BACKUP:',BACKUP,'MiB',round(BACKUP.stat().st_size/1024**2,1))
print('SHA256:',sha(BACKUP))
print('Download in JupyterLab before terminating your Runpod Pod')
